In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import argparse
from pathlib import Path

class MinecraftDatasetResizer:
    def __init__(self, input_dir, output_dir, target_size=224):
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.target_size = target_size
        self.supported_formats = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
        
    def create_output_structure(self):
        """Crea la estructura de carpetas de salida"""
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Copia la estructura de carpetas del input
        for item in self.input_dir.rglob('*'):
            if item.is_dir():
                relative_path = item.relative_to(self.input_dir)
                (self.output_dir / relative_path).mkdir(parents=True, exist_ok=True)
    
    def center_crop_square(self, image):
        """Hace un crop cuadrado desde el centro de la imagen"""
        h, w = image.shape[:2]
        
        # Determina el tamaño del cuadrado (el menor de alto o ancho)
        size = min(h, w)
        
        # Calcula las coordenadas del centro
        center_x, center_y = w // 2, h // 2
        
        # Calcula las coordenadas del crop
        x1 = center_x - size // 2
        y1 = center_y - size // 2
        x2 = x1 + size
        y2 = y1 + size
        
        return image[y1:y2, x1:x2]
    
    def resize_image(self, image_path):
        """Procesa una imagen individual"""
        try:
            # Lee la imagen
            image = cv2.imread(str(image_path))
            if image is None:
                print(f"❌ No se pudo leer: {image_path}")
                return False
            
            # Hace crop cuadrado central
            cropped = self.center_crop_square(image)
            
            # Redimensiona a tamaño objetivo
            resized = cv2.resize(cropped, (self.target_size, self.target_size), 
                               interpolation=cv2.INTER_LANCZOS4)
            
            # Calcula la ruta de salida
            relative_path = image_path.relative_to(self.input_dir)
            output_path = self.output_dir / relative_path
            
            # Guarda la imagen procesada
            cv2.imwrite(str(output_path), resized)
            
            return True
            
        except Exception as e:
            print(f"❌ Error procesando {image_path}: {str(e)}")
            return False
    
    def process_dataset(self):
        """Procesa todo el dataset"""
        print(f"🎮 Procesando dataset de Minecraft...")
        print(f"📁 Input: {self.input_dir}")
        print(f"📁 Output: {self.output_dir}")
        print(f"🎯 Tamaño objetivo: {self.target_size}x{self.target_size}")
        print("-" * 50)
        
        # Crea estructura de carpetas
        self.create_output_structure()
        
        # Encuentra todas las imágenes (evita duplicados)
        image_files = set()  # Usa set para evitar duplicados
        for ext in self.supported_formats:
            image_files.update(self.input_dir.rglob(f'*{ext}'))
            image_files.update(self.input_dir.rglob(f'*{ext.upper()}'))
        
        image_files = list(image_files)  # Convierte de vuelta a lista
        
        if not image_files:
            print("❌ No se encontraron imágenes en el directorio")
            return
        
        print(f"🔍 Encontradas {len(image_files)} imágenes")
        
        # Procesa cada imagen
        processed = 0
        failed = 0
        
        for i, image_path in enumerate(image_files, 1):
            print(f"Procesando {i}/{len(image_files)}: {image_path.name}", end=" ")
            
            if self.resize_image(image_path):
                print("✅")
                processed += 1
            else:
                failed += 1
        
        print("-" * 50)
        print(f"✅ Procesadas correctamente: {processed}")
        print(f"❌ Fallos: {failed}")
        print(f"🎯 Dataset redimensionado guardado en: {self.output_dir}")
    
    def preview_sample(self, num_samples=3):
        """Muestra preview de algunas imágenes antes de procesar"""
        print("🖼️  Preview del proceso de redimensionamiento:")
        
        image_files = []
        for ext in self.supported_formats:
            image_files.extend(self.input_dir.rglob(f'*{ext}'))
            if len(image_files) >= num_samples:
                break
        
        for i, image_path in enumerate(image_files[:num_samples]):
            image = cv2.imread(str(image_path))
            if image is not None:
                h, w = image.shape[:2]
                cropped = self.center_crop_square(image)
                ch, cw = cropped.shape[:2]
                print(f"  📷 {image_path.name}: {w}x{h} → {cw}x{ch} → {self.target_size}x{self.target_size}")


def main():
    parser = argparse.ArgumentParser(description='Redimensiona dataset de mobs de Minecraft para CNN')
    parser.add_argument('input_dir', help='Directorio con imágenes originales')
    parser.add_argument('output_dir', help='Directorio de salida')
    parser.add_argument('--size', type=int, default=224, 
                       help='Tamaño objetivo (default: 224)')
    parser.add_argument('--preview', action='store_true', 
                       help='Solo muestra preview sin procesar')
    
    args = parser.parse_args()
    
    resizer = MinecraftDatasetResizer(args.input_dir, args.output_dir, args.size)
    
    if args.preview:
        resizer.preview_sample()
    else:
        resizer.process_dataset()


if __name__ == "__main__":
    # Ejemplo de uso directo (sin argumentos de línea de comandos)
    # Modifica estas rutas según tu estructura
    INPUT_DIR = "C:/Users/Janus/OneDrive - IUE/SEMESTRE 2025-2/IA II/Dataset/Imagenes originales/creepers"  
    OUTPUT_DIR = "C:/Users/Janus/OneDrive - IUE/SEMESTRE 2025-2/IA II/Dataset/Imagenes redimensionadas/creepers"  
    TARGET_SIZE = 224                  # Tamaño final (224x224)
    
    resizer = MinecraftDatasetResizer(INPUT_DIR, OUTPUT_DIR, TARGET_SIZE)
    
    # Uncomment para ver preview antes de procesar
    # resizer.preview_sample()
    
    # Procesa el dataset completo
    resizer.process_dataset()

🎮 Procesando dataset de Minecraft...
📁 Input: C:\Users\Janus\OneDrive - IUE\SEMESTRE 2025-2\IA II\Dataset\Imagenes originales\creepers
📁 Output: C:\Users\Janus\OneDrive - IUE\SEMESTRE 2025-2\IA II\Dataset\Imagenes redimensionadas\creepers
🎯 Tamaño objetivo: 224x224
--------------------------------------------------
🔍 Encontradas 573 imágenes
Procesando 1/573: 2025-09-21_21.48.05 (1).png ✅
Procesando 2/573: 2025-09-21_21.38.12.png ✅
Procesando 3/573: 2025-09-21_21.16.52.png ✅
Procesando 4/573: 2025-09-21_21.17.22.png ✅
Procesando 5/573: 2025-09-21_21.31.58.png ✅
Procesando 6/573: 2025-09-21_21.48.50.png ✅
Procesando 7/573: 2025-09-21_21.52.48.png ✅
Procesando 8/573: 2025-09-21_21.37.13.png ✅
Procesando 9/573: 2025-09-21_21.16.08.png ✅
Procesando 10/573: 2025-09-21_21.53.35.png ✅
Procesando 11/573: 2025-09-21_21.22.45.png ✅
Procesando 12/573: 2025-09-21_21.22.19.png ✅
Procesando 13/573: 2025-09-21_21.15.40.png ✅
Procesando 14/573: 2025-09-21_21.37.58.png ✅
Procesando 15/573: 2025-09-21_2